# Data processes

The following scripts were all run via one of two remote servers--the ITKIN server, which was generously donated by Mark Itkin to the CoMind Lab, and my own personal server.

Each heading precedes the script run to complete that process/step. If one is running this via AWS or some other software, one just has to change to the data path parameters at the top of each script.

## Create COS values

In [ ]:
import pandas as pd
import torch
import numpy as np
import os
from datetime import datetime as dt
from convergence_entropy.CEDA.calc._bypass._experimental.deconstructed_cos import processor




###########################################################################################
# Basic set-up
###########################################################################################
print('CUDA:', torch.cuda.is_available())

DEV = 'cuda'

start = dt.now()
PATH = '/home/zprosen/d/shapeoflang3/'

RAW_PATH = os.path.join(PATH, 'server_ready')

CKPTS_PATH = os.path.join(PATH, 'ckpts')
if not os.path.exists(CKPTS_PATH):
    os.mkdir(CKPTS_PATH)

completed = [None]
if os.path.exists(os.path.join(PATH,'completed-null.txt')):
    completed = [l.strip() for l in open(os.path.join(PATH,'completed.txt'), 'r').readlines()]
else:
    f = open(os.path.join(PATH,'completed-null.txt'), 'w')
    f.write('')
    f.close()

level = [7,-1]

print(PATH, '\n', completed[-1], '\n', level, '\n\n')




############################################################################
# Main script
############################################################################

############################################################################
### Data set-up
############################################################################
subids = np.array([
    f for f in sorted(os.listdir(RAW_PATH))
    if (not f.startswith('.'))
       and (not os.path.isdir(os.path.join(RAW_PATH, f)))
], dtype='object')




#############################################################################
### starting from a saved checkpoint!
#############################################################################
print(subids, '\n\n')




#############################################################################
### Process
#############################################################################
with torch.no_grad():
    for i, submission_id in enumerate(subids):
        try:
            ############# import file
            if submission_id.endswith('.csv'):
                df = pd.read_csv(
                    os.path.join(RAW_PATH, submission_id)
                )

            elif submission_id.endswith('.parquet'):
                df = pd.read_parquet(
                    os.path.join(RAW_PATH, submission_id)
                )

            else:
                df = pd.read_table(
                    os.path.join(RAW_PATH, submission_id),
                    sep='\t'
                )

            ########## ID if xling file or not
            # if 'xling-' in submission_id:
            #     wv_model_name = 'xlm-roberta-base'
            # else:
            #     wv_model_name = 'roberta-base'
            wv_model_name = 'xlm-roberta-base'

            df = df.loc[
                (~df['x_utterance'].isna())
                & (~df['y_utterance'].isna())
            ]

            meta_data_cols = [col for col in list(df) if col not in ['x_utterance', 'y_utterance']]

            print('===]{}[==='.format(submission_id))
            print('{}/{}'.format(i+1, len(subids)))

            print(len(df), 'total contextual units')

            if 'conversation_id' in df.columns:
                print(df['conversation_id'].nunique(), 'conversations')

            groups = df.groupby(by=['corpus', 'conversation_id'])
            for labels, dfi in groups:

                checkpoint_name = ('xling-' * ('xling-' in submission_id)) + labels[0] + '-' + labels[1]

                if len(dfi) > 0:
                    # if checkpoint_name not in completed:
                    try:

                        GRAPH = processor(
                            wv_model=wv_model_name,
                            wv_layers=level,
                            device=DEV,
                            checkpoint_location=os.path.join(CKPTS_PATH, checkpoint_name)
                        )

                        GRAPH.fit(
                            x=dfi['x_utterance'].values,
                            y=dfi['y_utterance'].values,
                        )

                    except Exception as err:

                        torch.cuda.empty_cache()

                        err_path = os.path.join(PATH, 'ERRORs.txt')

                        if os.path.exists(err_path):
                            f  = open(err_path, 'a')
                            f.write(str(checkpoint_name)+ '\t' + str(err) + '\n')
                            f.close()

                        # else:
                        #     f = open(err_path, 'w')
                        #     f.write(str(submission_id)+ ' ' + str(err) + '\n')
                        #     f.close()

                f = open(os.path.join(PATH, 'completed-null.txt'), 'a')
                f.write(checkpoint_name + '\n')
                f.close()

        except Exception:
            0

print('=======][=======\n')


## Convert CoS to Information

In [ ]:
###############################################################
#### ISSUES
# (1) How to make sure that conversation id is (a) in corpus and (b) aligned correctly.
# Meta data files have format like this: 'CANDOR-111-null_perm.parquet',
# while data containing files have format like this:
# 'CANDOR-CANDOR-26-c7b2bf7c-ed3b-4adf-82b4-c9232cdffca9'.

# Solution: load meta data files.
###############################################################
import pandas as pd
import torch
import numpy as np
import os
from datetime import datetime as dt
from tqdm import tqdm
from convergence_entropy.CEDA.calc._bypass._experimental.deconstructed_cos import processor
import warnings; warnings.filterwarnings('ignore')

############################################################################
# Basic set-up
############################################################################
print('CUDA:', torch.cuda.is_available())

DEV = 'cuda'

start = dt.now()
DATA_PATH = '/home/zprosen/d/shapeoflang3/'

RAW_PATH = os.path.join(DATA_PATH, 'server_ready')

CKPTS_PATH = os.path.join(DATA_PATH, 'ckpts')

PROCESSED_PATH = os.path.join(CKPTS_PATH, 'lme-ready')
if not os.path.exists(PROCESSED_PATH):
    os.mkdir(PROCESSED_PATH)

completed = [None]

meta_data_cols = ['conversation_id', 'corpus', 'x_speaker', 'y_speaker', 'x_turn_id', 'y_turn_id']

completed = [l.strip() for l in open(os.path.join(DATA_PATH,'completed-checklist.txt'), 'r').readlines()]

level = [7,-1]

print(RAW_PATH, DATA_PATH, PROCESSED_PATH, '\n', completed[-1], '\n', level, '\n\n')




############################################################################
# Post-processing-function
############################################################################
sigma = .3
N = torch.distributions.HalfNormal(scale=sigma)

# Gausian/Cooney Kernel
def I(x, sigma=sigma):
    x_ = x + 1e-9
    p = np.exp(- (x_**2)/(2*(sigma**2)) )
    return (- np.log( p )).sum().item()

# Convergence-Entropy (current version)
def H(x):
    p = 1-N.cdf(torch.FloatTensor(x))
    return - (p * torch.log(p)).sum().item()




############################################################################
# Main script
############################################################################

############################################################################
### Data set-up
############################################################################
# subids = np.array(completed, dtype='object')
CORPUS_DOCS = [f.replace('.parquet', '') for f in os.listdir(RAW_PATH) if (not f.startswith('.')) and (f != 'lme-ready') and (f != 'to_itkin') and (f != 'to_rosen')]
conversations = [f for f in os.listdir(CKPTS_PATH) if (not f.startswith('.')) and (f != 'lme-ready') and (f != 'to_itkin') and (f != 'to_rosen')]
CORPUS_DOCS = ['xling-'+ f if f in set(CORPUS_DOCS).difference(set(conversations)) else f for f in CORPUS_DOCS]
print(set(conversations).difference(set(CORPUS_DOCS)))
CORPUS_DOCS = [f + '.parquet' for f in CORPUS_DOCS]


#############################################################################
### Process
#############################################################################
with torch.no_grad():

    for CORPUS in CORPUS_DOCS:
        print(CORPUS)
        CID = CORPUS.replace('.parquet', '')

        outfile = os.path.join(CKPTS_PATH, 'lme-ready', CID + '.parquet')

        try:
            meta_data = pd.read_parquet(
                os.path.join(RAW_PATH, CORPUS),
                columns=meta_data_cols
            )

        except Exception:
            meta_data = pd.read_parquet(
                os.path.join(RAW_PATH, CORPUS).replace('xling-', ''),
                columns=meta_data_cols
            )

        conversation = os.path.join(CKPTS_PATH, CID)
        conversation = [os.path.join(CKPTS_PATH, CID, fi) for fi in os.listdir(conversation)]
        conversation = np.array(
            [[fi, int(fi.split('-')[-1].replace('.parquet', ''))] for fi in conversation],
            dtype=object
        )
        conversation = conversation[:,0][conversation[:, 1].argsort()]

        df = pd.concat([pd.read_parquet(fi) for fi in conversation], ignore_index=True)

        df[['aCoS', 'I', 'H']] = 0

        for i in tqdm(df.index):
            nx = int(df['nx'].loc[i])
            # x = np.array(df['CoS'].loc[i].replace('[', '').replace(']', '').split(', '), dtype=float)
            x = np.array(df['CoS'].loc[i], dtype=float)
            x = 1 - np.where(x > .9999999, .9999999, x)
            m = x.mean().item()
            x = x[:nx]
            # x = torch.FloatTensor(x)

            df.loc[i, ['aCoS', 'I', 'H']] = [m, I(x), H(x)]

        del df['CoS']

        df.index=range(len(df))
        meta_data.index=range(len(meta_data))
        # df = df.join(meta_data)
        df = pd.concat([meta_data, df], axis=1)

        df.to_parquet(
            outfile,
            engine='fastparquet',
            compression='snappy'
        )

        del df

        print('====][====')

print('=======]fin[=======\n')


## Add AVAR ($\sigma^2(\tau)$) Column

_Needs to be revised so that subids is actually created from sampling the files in the folder `.../ckpts/lme-ready`_

In [ ]:
import gc
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
import warnings; warnings.filterwarnings('ignore')

DATA_PATH = '/home/zprosen/d/shapeoflang3/'
LME_READY_DIR = os.path.join(DATA_PATH, 'ckpts', 'lme-ready')
CHECKLIST_FILE = os.path.join(DATA_PATH, 'completed.txt')

turn_col = 'x_turn_id'
conversation_id_col = 'conversation_id'
ordinal_col = 'sample_delta'
val_col = 'I'

merge_cols = [conversation_id_col, 'self', turn_col]

f = open(os.path.join(DATA_PATH,CHECKLIST_FILE), 'r')
subids = np.array([fi.strip() for fi in f.readlines() if fi.strip()], dtype=object)



####################################################################
# AVAR fn
####################################################################
def add_avar_col(df, val_col, merge_cols):
    # differentiate turn id cols
    df[turn_col] = df[conversation_id_col].astype(str) + '-' + df[turn_col].astype(str)

    # calculate period length
    tau = df[merge_cols].value_counts().reset_index()
    tau = tau.rename(columns={'count': 'tau'})

    # add period length col
    df = pd.merge(
        left=df, left_on=merge_cols,
        right=tau, right_on=merge_cols,
        how='left'
    )

    # delete tau df and collect memory
    del tau
    gc.collect()

    # add next step in series
    df[val_col + '_2'] = None
    df[val_col + '_2'].loc[:len(df) - 2] = df[val_col].values[1:]

    # add next, next step in series
    df[val_col + '_3'] = None
    df[val_col + '_3'].loc[:len(df) - 3] = df[val_col].values[2:]

    # calculate AVAR
    df['AVAR'] = (1 / (2 * (df['tau'] ** 2))) * (df[val_col + '_3'] - (2 * df[val_col + '_2']) + df[val_col]) ** 2

    # render comparisons to values that are otherwise erroneous null
    sel = []
    for col in merge_cols:
        v = df[col].values
        sel += [((v[:-1] == v[1:])[:-1] & (v[:-2] == v[2:])).astype(int).reshape(-1,1)]

        # # collect excess memory
        # del v
        # gc.collect()

    sel = np.concat(sel,axis=1).sum(axis=-1) != len(merge_cols)
    sel = np.concat([sel, [False] * 2])

    # df = df.loc[:len(df)-3]
    df['AVAR'].loc[sel] = None

    # collect memory
    del sel
    del df[val_col + '_2']
    del df[val_col + '_3']
    gc.collect()

    return df




####################################################################
# The show
####################################################################
for f in tqdm(subids):
    f_ = os.path.join(LME_READY_DIR, f)
    df = pd.read_parquet(f_, engine='fastparquet')
    df['self'] = df['x_speaker'] == df['y_speaker']
    df = add_avar_col(df, val_col=val_col, merge_cols=merge_cols)

    # if doing regression outright, do:
    #

    # if saving AVAR for later, do:
    df.to_parquet(
        f_,
        engine='fastparquet',
        compression='snappy'
    )